In [1]:
import sys, os
sys.path.append("../")

import jax
jax.config.update("jax_enable_x64", True)

import lineax as lx

from qd_solve import *
from qd_solve.operator import *
from qd_solve.spaces.pseudospectral import *
from qd_solve.spaces.finite_difference import *
from qd_solve.exp import *

from miscutils.plot import animate

import jax.numpy as jnp
import diffrax

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import Image

In [41]:
x0 = -10
xf = 10
num_steps = 1000
num_modes = 250

space = PseudoSpectral(x0, xf, num_steps, num_modes)
fn = lambda x: 0.5 * x ** 2
V = PseudoSpectralPotentialEnergy(fn).set_exponentiator(ExactExponentiator())
V_cn = PseudoSpectralPotentialEnergy(fn).set_exponentiator(KrylovExponentiator(250))

V_p = PseudoSpectralPotentialEnergy(fn).set_exponentiator(PadeExponentiator(m=50, s=5, mu=0.0))


V_cheby = V.set_exponentiator(ChebyshevExponentiator(35))

In [42]:
key = jax.random.key(0)
y_vals = jax.random.normal(key, shape=(num_steps,), dtype=jnp.array(1j).dtype)
y = space.from_values(y_vals)

In [43]:
dt = 0.1

jnp.linalg.norm((V_p.exp(dt, y) - V.exp(dt, y)).coeffs), jnp.linalg.norm((V_cheby.exp(dt, y) - V.exp(dt, y)).coeffs)

(Array(16.49492838, dtype=float64), Array(0.87325024, dtype=float64))